# Model Scoring
#### Make the files with model probabilities ready for Kaggle submission 

In [1]:
# Import necessary modules
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import os
from os import listdir
from os.path import isfile, join
from PIL import Image
import keras
from keras import layers
import csv
from tensorflow.keras.utils import to_categorical
import tensorflow as tf

2025-11-08 07:06:46.026673: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762585606.240683      48 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762585606.304069      48 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
# Read all .tif files from the 'test' folder
test_obs = []
for dirname, _, filenames in os.walk('/kaggle/input/histopathologic-cancer-detection/test'):
    for filename in filenames:
        test_obs.append(filename)

In [21]:
# Cut off the .tif part in the file names  
test_obs_df = []
for i in test_obs:
    test_obs_df.append(i[:40])

# Turn list with adjusted file names in a data frame
id_df = pd.DataFrame(test_obs_df, columns=['id'])

### RMSProp baseline version

In [54]:
# Read file with predicted probabilities from CNN with optimizer RMSProp
rms_outcome = pd.read_csv('rmsprop.csv')
rms_outcome

,0,1,Label
0,0.009837,9.901629e-01,1
1,0.153352,8.466479e-01,1
2,0.001013,9.989874e-01,1
3,0.555589,4.444108e-01,0
4,0.000208,9.997916e-01,1
...,...,...,...
57453,0.986024,1.397630e-02,0
57454,0.002193,9.978074e-01,1
57455,0.000085,9.999147e-01,1
57456,1.000000,2.077614e-08,0


In [55]:
# Adjust format so it can be submitted to Kaggle
rms_outcome_def = pd.concat([id_df, rms_outcome], axis=1, join="inner").loc[:,['id','Label']]
rms_outcome_def.rename(columns={'Label':'label'}, inplace = True)
rms_outcome_def

,id,label
0,a7ea26360815d8492433b14cd8318607bcf99d9e,1
1,59d21133c845dff1ebc7a0c7cf40c145ea9e9664,1
2,5fde41ce8c6048a5c2f38eca12d6528fa312cdbb,1
3,bd953a3b1db1f7041ee95ff482594c4f46c73ed0,0
4,523fc2efd7aba53e597ab0f69cc2cbded7a6ce62,1
...,...,...
57453,7907c88a7f5f9c8ca5b2df72c1e6ff9650eea22b,0
57454,2a6fc1ed16fa94d263efab330ccbeb1906cbd421,1
57455,6bb5c0611c0ccf4713e0ccbc0e8c54bcb498ef14,1
57456,f11e7c9e77cbc1ec916a52e6b871a293ee1bb928,0


In [53]:
# Write rms_outcome_def to external location
rms_outcome_def.to_csv('rms_outcome_def.csv', index=False)

### RMSProp (decision threshold of 0.1)

In [57]:
# Rename columns of rms_outcome
rms_outcome.rename(columns= {'0': 'zero', '1': 'one'},inplace= True)
rms_outcome

# Add columns label_I with a decision threshold of 0.1 (instead of 0.5)
rms_outcome['Label_I'] = [1 if score >= 0.10 else 0 for score in rms_outcome['one']]
rms_outcome

,zero,one,Label,Label_I
0,0.009837,9.901629e-01,1,1
1,0.153352,8.466479e-01,1,1
2,0.001013,9.989874e-01,1,1
3,0.555589,4.444108e-01,0,1
4,0.000208,9.997916e-01,1,1
...,...,...,...,...
57453,0.986024,1.397630e-02,0,0
57454,0.002193,9.978074e-01,1,1
57455,0.000085,9.999147e-01,1,1
57456,1.000000,2.077614e-08,0,0


In [59]:
# Adjust format so it can be submitted to Kaggle
rms_outcome_def_I = pd.concat([id_df, rms_outcome], axis=1, join="inner").loc[:,['id','Label_I']]
rms_outcome_def_I.rename(columns= {'Label_I': 'label'},inplace= True)
rms_outcome_def_I

,id,label
0,a7ea26360815d8492433b14cd8318607bcf99d9e,1
1,59d21133c845dff1ebc7a0c7cf40c145ea9e9664,1
2,5fde41ce8c6048a5c2f38eca12d6528fa312cdbb,1
3,bd953a3b1db1f7041ee95ff482594c4f46c73ed0,1
4,523fc2efd7aba53e597ab0f69cc2cbded7a6ce62,1
...,...,...
57453,7907c88a7f5f9c8ca5b2df72c1e6ff9650eea22b,0
57454,2a6fc1ed16fa94d263efab330ccbeb1906cbd421,1
57455,6bb5c0611c0ccf4713e0ccbc0e8c54bcb498ef14,1
57456,f11e7c9e77cbc1ec916a52e6b871a293ee1bb928,0


In [60]:
# Write rms_outcome_def_I to external location
rms_outcome_def_I.to_csv('rms_outcome_def_I.csv', index=False)

### Adam Optimizer Batch 400 baseline version

In [34]:
# Read file with predicted probabilities from CNN with optimizer Adam and batch 400
adam_outcome = pd.read_csv('adamb400.csv')
adam_outcome

,0,1,Label
0,0.184185,8.158150e-01,1
1,0.999468,5.317341e-04,0
2,0.981081,1.891942e-02,0
3,0.997306,2.694035e-03,0
4,0.999994,5.750564e-06,0
...,...,...,...
57453,0.998094,1.906083e-03,0
57454,1.000000,3.224339e-09,0
57455,0.780777,2.192231e-01,0
57456,1.000000,2.728337e-16,0


In [39]:
# Adjust format so it can be submitted to Kaggle
adam_outcome_def = pd.concat([id_df, adam_outcome], axis=1, join="inner").loc[:,['id','Label']]
adam_outcome_def.rename(columns={'Label':'label'}, inplace = True)
adam_outcome_def

,id,label
0,a7ea26360815d8492433b14cd8318607bcf99d9e,1
1,59d21133c845dff1ebc7a0c7cf40c145ea9e9664,0
2,5fde41ce8c6048a5c2f38eca12d6528fa312cdbb,0
3,bd953a3b1db1f7041ee95ff482594c4f46c73ed0,0
4,523fc2efd7aba53e597ab0f69cc2cbded7a6ce62,0
...,...,...
57453,7907c88a7f5f9c8ca5b2df72c1e6ff9650eea22b,0
57454,2a6fc1ed16fa94d263efab330ccbeb1906cbd421,0
57455,6bb5c0611c0ccf4713e0ccbc0e8c54bcb498ef14,0
57456,f11e7c9e77cbc1ec916a52e6b871a293ee1bb928,0


In [41]:
# Write adam_outcome_def to external location
adam_outcome_def.to_csv('adam_outcome_def.csv', index=False)

### Adam Optimizer Batch 400 (decision threshold of 0.1)

In [62]:
# Rename columns of adam_outcome
adam_outcome.rename(columns= {'0': 'zero', '1': 'one'},inplace= True)
adam_outcome

# Add columns label_I with a decision threshold of 0.1 (instead of 0.5)
adam_outcome['Label_I'] = [1 if score >= 0.10 else 0 for score in adam_outcome['one']]
adam_outcome

,zero,one,Label,Label_I
0,0.184185,8.158150e-01,1,1
1,0.999468,5.317341e-04,0,0
2,0.981081,1.891942e-02,0,0
3,0.997306,2.694035e-03,0,0
4,0.999994,5.750564e-06,0,0
...,...,...,...,...
57453,0.998094,1.906083e-03,0,0
57454,1.000000,3.224339e-09,0,0
57455,0.780777,2.192231e-01,0,1
57456,1.000000,2.728337e-16,0,0


In [63]:
# Adjust format so it can be submitted to Kaggle
adam_outcome_def_I = pd.concat([id_df, adam_outcome], axis=1, join="inner").loc[:,['id','Label_I']]
adam_outcome_def_I.rename(columns={'Label_I':'label'}, inplace = True)
adam_outcome_def_I

,id,label
0,a7ea26360815d8492433b14cd8318607bcf99d9e,1
1,59d21133c845dff1ebc7a0c7cf40c145ea9e9664,0
2,5fde41ce8c6048a5c2f38eca12d6528fa312cdbb,0
3,bd953a3b1db1f7041ee95ff482594c4f46c73ed0,0
4,523fc2efd7aba53e597ab0f69cc2cbded7a6ce62,0
...,...,...
57453,7907c88a7f5f9c8ca5b2df72c1e6ff9650eea22b,0
57454,2a6fc1ed16fa94d263efab330ccbeb1906cbd421,0
57455,6bb5c0611c0ccf4713e0ccbc0e8c54bcb498ef14,1
57456,f11e7c9e77cbc1ec916a52e6b871a293ee1bb928,0


In [64]:
# Write adam_outcome_def_I to external location
adam_outcome_def_I.to_csv('adam_outcome_def_I.csv', index=False)